In [1]:
!pip install -q pandas httpx tqdm

In [2]:
import os
import re
import json
import asyncio
import inspect
import pandas as pd
import httpx

from tqdm.auto import tqdm
from google.colab import files

In [70]:
# ----------------------------
# Config
# ----------------------------
os.environ["OPENROUTER_API_KEY"] = ""   # replace with your real key
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_URL = "https://openrouter.ai/api/v1/chat/completions"

if not OPENROUTER_API_KEY or OPENROUTER_API_KEY == "key here":
    raise ValueError("Set OPENROUTER_API_KEY to your real OpenRouter key first.")

# Pick one valid model id
# model_id = "qwen/qwen3.6-plus"
model_id = "xiaomi/mimo-v2-flash"
# model_id = "google/gemma-4-26b-a4b-it"
# model_id = "mistralai/mistral-small-2603"
# model_id = "openai/gpt-5.4-nano"
# model_id = "x-ai/grok-4.20-multi-agent"
# model_id = "deepseek/deepseek-v3.2"
# model_id = "anthropic/claude-3.5-haiku"

In [71]:
# ----------------------------
# Upload + load CSV
# ----------------------------
uploaded = files.upload()
if not uploaded:
    raise ValueError("No file uploaded.")

uploaded_filename = next(iter(uploaded.keys()))
df = pd.read_csv(uploaded_filename)

if inspect.iscoroutine(df):
    raise TypeError("df is a coroutine. Reload the CSV and do not assign async function results to df unless you use await.")

if not isinstance(df, pd.DataFrame):
    raise TypeError(f"Expected pandas DataFrame, got {type(df)}")

print("Loaded file:", uploaded_filename)
print("Columns:", list(df.columns))
display(df.head())

# Set these directly for your file
text_column = "Collected Text"
label_column = "Given Label"

if text_column not in df.columns:
    raise ValueError(f"Text column '{text_column}' not found. Available columns: {list(df.columns)}")

if label_column not in df.columns:
    raise ValueError(f"Label column '{label_column}' not found. Available columns: {list(df.columns)}")

print("Using text column:", text_column)
print("Using label column:", label_column)

Saving Reactions - 500 - Sheet1.csv to Reactions - 500 - Sheet1 (24).csv
Loaded file: Reactions - 500 - Sheet1 (24).csv
Columns: ['Collected Text', 'Given Label']


,Collected Text,Given Label
0,Y'all only know it's AI because of how bad it is.,Negative
1,Normalise ai use everyone uses it. your google...,Positive
2,Man AI really looks good ..,Positive
3,this is weird ai should be banned,Negative
4,You can’t tell this is AI,Neutral


Using text column: Collected Text
Using label column: Given Label


In [72]:
# ----------------------------
# Helpers
# ----------------------------
def normalize_sentiment(value: str) -> str:
    if not isinstance(value, str):
        return "Unknown"

    text = value.strip().lower()

    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict) and "sentiment" in parsed:
            text = str(parsed["sentiment"]).strip().lower()
    except Exception:
        pass

    if text in {"positive", '"positive"'}:
        return "Positive"
    if text in {"negative", '"negative"'}:
        return "Negative"
    if text in {"neutral", '"neutral"'}:
        return "Neutral"

    if "positive" in text:
        return "Positive"
    if "negative" in text:
        return "Negative"
    if "neutral" in text:
        return "Neutral"

    return "Unknown"


def sanitize_model_name(model_id: str) -> str:
    return re.sub(r"[^A-Za-z0-9_]+", "_", model_id).strip("_").lower()


async def call_openrouter_model(client, model_id: str, text: str) -> str:
    payload = {
        "model": model_id,
        "messages": [
            {
                "role": "system",
                "content": "You are a sentiment classifier. Return exactly one word: Positive, Negative, or Neutral."
            },
            {
                "role": "user",
                "content": f"Classify the sentiment of this text: {text}"
            }
        ],
        "temperature": 0,
        "max_completion_tokens": 3,
    }

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    response = await client.post(
        OPENROUTER_API_URL,
        headers=headers,
        json=payload,
    )
    response.raise_for_status()
    data = response.json()
    content = data["choices"][0]["message"]["content"].strip()
    return normalize_sentiment(content)


async def run_one_model(df, model_id: str, text_column: str, concurrency: int = 8):
    if inspect.iscoroutine(df):
        raise TypeError(
            "run_one_model received a coroutine instead of a DataFrame. "
            "Reload df from pd.read_csv(...) and ensure previous async calls used await."
        )
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"run_one_model expected DataFrame, got {type(df)}")
    if text_column not in df.columns:
        raise ValueError(f"Text column '{text_column}' not found. Available columns: {list(df.columns)}")

    sentiment_col = f"{sanitize_model_name(model_id)}_sentiment"
    error_col = f"{sanitize_model_name(model_id)}_error"

    if sentiment_col not in df.columns:
        df[sentiment_col] = None
    if error_col not in df.columns:
        df[error_col] = None

    semaphore = asyncio.Semaphore(concurrency)
    timeout = httpx.Timeout(60.0, connect=20.0)
    limits = httpx.Limits(
        max_connections=max(10, concurrency * 2),
        max_keepalive_connections=max(10, concurrency),
    )

    async with httpx.AsyncClient(timeout=timeout, limits=limits) as client:
        async def worker(idx, text):
            async with semaphore:
                if not isinstance(text, str) or not text.strip():
                    return idx, "No Text", ""

                try:
                    sentiment = await call_openrouter_model(client, model_id, text)
                    return idx, sentiment, ""
                except Exception as e:
                    return idx, "Error", str(e)

        tasks = [
            asyncio.create_task(worker(i, str(text) if pd.notna(text) else ""))
            for i, text in enumerate(df[text_column].tolist())
        ]

        for task in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc=model_id):
            idx, sentiment, error = await task
            df.at[idx, sentiment_col] = sentiment
            df.at[idx, error_col] = error

    return df

In [73]:
# ----------------------------
# Run model
# ----------------------------
print("df type before run_one_model:", type(df))

df = await run_one_model(
    df,
    model_id=model_id,
    text_column=text_column,
    concurrency=8,
)

print("df type after run_one_model:", type(df))
display(df.head())

df type before run_one_model: <class 'pandas.core.frame.DataFrame'>


xiaomi/mimo-v2-flash:   0%|          | 0/499 [00:00<?, ?it/s]

df type after run_one_model: <class 'pandas.core.frame.DataFrame'>


,Collected Text,Given Label,xiaomi_mimo_v2_flash_sentiment,xiaomi_mimo_v2_flash_error
0,Y'all only know it's AI because of how bad it is.,Negative,Negative,
1,Normalise ai use everyone uses it. your google...,Positive,Neutral,
2,Man AI really looks good ..,Positive,Positive,
3,this is weird ai should be banned,Negative,Negative,
4,You can’t tell this is AI,Neutral,Neutral,


In [74]:
# ----------------------------
# Save + download
# ----------------------------
output_filename = "sentiment_final_output.csv"
df.to_csv(output_filename, index=False)
files.download(output_filename)
print(f"Saved and downloaded: {output_filename}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved and downloaded: sentiment_final_output.csv
